In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

os.listdir('/content/drive/MyDrive/anemia_detection/test_dataset')

['palm_test.zip', 'conjunctiva.zip']

In [ ]:
!unzip "/content/drive/MyDrive/anemia_detection/test_dataset/conjunctiva.zip"

Archive:  /content/drive/MyDrive/anemia_detection/test_dataset/conjunctiva.zip
  inflating: dataset anemia/Dataset anemia.docx  
  inflating: dataset anemia/India/1/20200118_164733.jpg  
  inflating: dataset anemia/India/1/20200118_164733_forniceal.png  
  inflating: dataset anemia/India/1/20200118_164733_forniceal_palpebral.png  
  inflating: dataset anemia/India/1/20200118_164733_palpebral.png  
  inflating: dataset anemia/India/10/20200203_091841.jpg  
  inflating: dataset anemia/India/10/20200203_091841_forniceal.png  
  inflating: dataset anemia/India/10/20200203_091841_forniceal_palpebral.png  
  inflating: dataset anemia/India/10/20200203_091841_palpebral.png  
  inflating: dataset anemia/India/11/20200203_094523.jpg  
  inflating: dataset anemia/India/11/20200203_094523_forniceal.png  
  inflating: dataset anemia/India/11/20200203_094523_forniceal_palpebral.png  
  inflating: dataset anemia/India/11/20200203_094523_palpebral.png  
  inflating: dataset anemia/India/12/20200203_1

In [ ]:
import os
from glob import glob

df = "/content/dataset anemia"

# Tüm alt klasörlerdeki jpg ve png dosyalarını al
images = glob(df + "/**/*.jpg", recursive=True) + glob(df + "/**/*.png", recursive=True)

print("Toplam görüntü:", len(images))
print(images[:5])

Toplam görüntü: 862
['/content/dataset anemia/Italy/39/T_8_20190607_075130.jpg', '/content/dataset anemia/Italy/58/T_27_20190608_100451.jpg', '/content/dataset anemia/Italy/50/T_19_20190608_081908.jpg', '/content/dataset anemia/Italy/62/T_31_20190608_113406.jpg', '/content/dataset anemia/Italy/80/T_49_20190611_095655.jpg']


In [ ]:
import os
from glob import glob

images = glob(df + "/**/**/*")

print("Toplam bulunan dosya:", len(images))
for x in images[:10]:
    print(repr(os.path.basename(x)))

Toplam bulunan dosya: 862
'T_8_20190607_075130_forniceal_palpebral.png'
'T_8_20190607_075130.jpg'
'T_8_20190607_075130_forniceal.png'
'T_8_20190607_075130_palpebral.png'
'T_27_20190608_100451.jpg'
'T_27_20190608_100451_palpebral.png'
'T_19_20190608_081908.jpg'
'T_19_20190608_081908_forniceal.png'
'T_19_20190608_081908_forniceal_palpebral.png'
'T_19_20190608_081908_palpebral.png'


In [ ]:
import pandas as pd
import numpy as np

# Dosya yolları
files = [
    "/content/dataset anemia/India/India.xlsx",
    "/content/dataset anemia/Italy/Italy.xlsx"
]

for file in files:
    print(f"\n--- {file} ---")
    df = pd.read_excel(file)

    # Hgb ve Age sütunlarını sayıya çevir
    df['Hgb'] = pd.to_numeric(df['Hgb'].astype(str).str.replace(',', '.'), errors='coerce')
    df['Age'] = pd.to_numeric(df['Age'], errors='coerce')

    # Eksik değer sayısı
    missing = df[['Hgb', 'Gender', 'Age']].isna().sum()
    print("Eksik değer sayısı:")
    print(missing)

    # Eksik değerleri doldur (artık inplace kullanmıyoruz)
    df['Hgb'] = df['Hgb'].fillna(df['Hgb'].mean())
    df['Age'] = df['Age'].fillna(df['Age'].mean())
    df['Gender'] = df['Gender'].fillna('Unknown')

    # İlk 5 satırı göster
    print("\nSütun örnekleri:")
    print(df[['Hgb', 'Gender', 'Age']].head())


--- /content/dataset anemia/India/India.xlsx ---
Eksik değer sayısı:
Hgb       0
Gender    0
Age       0
dtype: int64

Sütun örnekleri:
    Hgb Gender  Age
0  12.2      M   29
1   8.0      F   36
2  10.7      F   30
3   8.3      F   39
4   7.8      F   29

--- /content/dataset anemia/Italy/Italy.xlsx ---
Eksik değer sayısı:
Hgb       1
Gender    0
Age       0
dtype: int64

Sütun örnekleri:
    Hgb Gender  Age
0   9.3      F   82
1  10.2      F   77
2  10.7      F   52
3  11.7      F   73
4  11.6      F   74


In [ ]:
import pandas as pd
import numpy as np

# Dosya yolları
india_path = "/content/dataset anemia/India/India.xlsx"
italy_path = "/content/dataset anemia/Italy/Italy.xlsx"

# Excel dosyalarını oku
india_df = pd.read_excel(india_path)
italy_df = pd.read_excel(italy_path)

# Number dahil gerekli sütunlar
cols = ["Number", "Hgb", "Gender", "Age"]
india_df = india_df[cols]
italy_df = italy_df[cols]

# Hgb değerlerini float yap
for df in [india_df, italy_df]:
    # Sayısal olmayanları NaN yap, virgülü noktaya çevir
    df['Hgb'] = pd.to_numeric(df['Hgb'].astype(str).str.replace(',', '.'), errors='coerce')

# Italy Hgb'deki eksik değerleri mean ile doldur
italy_df['Hgb'] = italy_df['Hgb'].fillna(italy_df['Hgb'].mean())
# Anemi hedef sütunu oluşturma fonksiyonu
def anemi_target(row):
    hgb = row['Hgb']
    age = row['Age']
    gender = row['Gender'].upper()

    if age < 0.25:  # Yenidoğan ~ 3 ay = 0.25 yıl
        low, high = 13.5, 24
    elif 0 <= age < 3:
        low, high = 11, 13
    elif 3 <= age < 11:
        low, high = 12, 14
    elif gender == 'M':
        low, high = 13.5, 17.5
    else:  # Kadın yetişkin
        low, high = 12.5, 15.5

    return "anemic" if hgb < low else "non anemic"

# target sütununu ekle
india_df['target'] = india_df.apply(anemi_target, axis=1)
italy_df['target'] = italy_df.apply(anemi_target, axis=1)

# Sonuçları kaydet
india_df.to_excel("/content/dataset anemia/India/India_processed.xlsx", index=False)
italy_df.to_excel("/content/dataset anemia/Italy/Italy_processed.xlsx", index=False)

print("İşlem tamamlandı. Dosyalar kaydedildi.")

İşlem tamamlandı. Dosyalar kaydedildi.


In [ ]:
import os
import pandas as pd

# İşlenmiş excel dosyaları
india_df = pd.read_excel("/content/dataset anemia/India/India_processed.xlsx")
italy_df = pd.read_excel("/content/dataset anemia/Italy/Italy_processed.xlsx")

# Görsel tipleri
image_types = [
    "forniceal_palpebral",
    "forniceal",
    "palpebral"
]

# Ana klasörler (burayı dataset yapına göre düzenle)
india_img_path = "/content/dataset anemia/India"
italy_img_path = "/content/dataset anemia/Italy"

def create_image_df(df, base_path, country_name):
    data = []

    for _, row in df.iterrows():
        number = row["Number"]
        label = row["target"]

        folder_path = os.path.join(base_path, str(number))

        if not os.path.exists(folder_path):
            continue

        for file in os.listdir(folder_path):
            if file.endswith(".png"):
                # sadece istediğimiz 3 tip
                if any(t in file for t in image_types):
                    full_path = os.path.join(folder_path, file)

                    data.append({
                        "Number": number,
                        "image_path": full_path,
                        "image_type": next((t for t in image_types if t in file), None),
                        "target": label,
                        "country": country_name
                    })

    return pd.DataFrame(data)

# DataFrame'leri oluştur
india_images_df = create_image_df(india_df, india_img_path, "india")
italy_images_df = create_image_df(italy_df, italy_img_path, "italy")

# Birleştir (isteğe bağlı ama öneririm)
all_images_df = pd.concat([india_images_df, italy_images_df], ignore_index=True)

# Kontrol
print(all_images_df.head())
print("Toplam görüntü:", len(all_images_df))

   Number                                         image_path  \
0       1  /content/dataset anemia/India/1/20200118_16473...   
1       1  /content/dataset anemia/India/1/20200118_16473...   
2       1  /content/dataset anemia/India/1/20200118_16473...   
3       2  /content/dataset anemia/India/2/20200124_15432...   
4       2  /content/dataset anemia/India/2/20200124_15432...   

            image_type  target country  
0            forniceal  anemic   india  
1  forniceal_palpebral  anemic   india  
2            palpebral  anemic   india  
3  forniceal_palpebral  anemic   india  
4            palpebral  anemic   india  
Toplam görüntü: 642


In [ ]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.8 MB/s eta 0:00:00


In [ ]:
df_test = all_images_df.copy()

df_test = df_test.rename(columns={
    "image_path": "path",
    "target": "class"
})

# formatı modele uygun hale getir
df_test["class"] = df_test["class"].map({
    "anemic": "Anemic",
    "non anemic": "Non-Anemic"
})

In [ ]:
import cv2
import numpy as np

features = []
labels = []

for _, row in df_test.iterrows():
    img = cv2.imread(row["path"])
    if img is None:
        continue

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)

    L, a, b = cv2.split(lab)

    mean_a = np.mean(a)
    mean_b = np.mean(b)
    mean_g = np.mean(img[:, :, 1])

    features.append([mean_a, mean_b, mean_g])

    if row["class"] == "Anemic":
        labels.append(1)
    else:
        labels.append(0)

X_test = np.array(features, dtype=np.float32)
y_test = np.array(labels, dtype=np.int32)

print("Test veri shape:", X_test.shape)

Test veri shape: (642, 3)


In [ ]:
import os
import joblib
from sklearn.metrics import accuracy_score, classification_report

model_path = "/content/drive/MyDrive/anemia_detection/saved_models/conjunctiva_models"

results = []

for file in os.listdir(model_path):
    if file.endswith(".joblib"):
        full_path = os.path.join(model_path, file)

        model = joblib.load(full_path)

        y_pred = model.predict(X_test)

        acc = accuracy_score(y_test, y_pred)

        print(f"\nModel: {file}")
        print("Accuracy:", acc)
        print(classification_report(y_test, y_pred))

        results.append((file, acc))


Model: Naive_Bayes.joblib
Accuracy: 0.42990654205607476
              precision    recall  f1-score   support

           0       0.30      0.09      0.14       328
           1       0.45      0.79      0.57       314

    accuracy                           0.43       642
   macro avg       0.38      0.44      0.36       642
weighted avg       0.38      0.43      0.35       642


Model: k-NN.joblib
Accuracy: 0.4688473520249221
              precision    recall  f1-score   support

           0       0.48      0.54      0.51       328
           1       0.45      0.40      0.42       314

    accuracy                           0.47       642
   macro avg       0.47      0.47      0.47       642
weighted avg       0.47      0.47      0.47       642


Model: XGBoost.joblib
Accuracy: 0.5109034267912772
              precision    recall  f1-score   support

           0       0.53      0.44      0.48       328
           1       0.50      0.59      0.54       314

    accuracy            

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



Model: LightGBM_model.joblib
Accuracy: 0.5109034267912772
              precision    recall  f1-score   support

           0       0.53      0.34      0.42       328
           1       0.50      0.68      0.58       314

    accuracy                           0.51       642
   macro avg       0.52      0.51      0.50       642
weighted avg       0.52      0.51      0.50       642

